# contiguous-layout — ex3: classify contiguity from shape and stride tuples

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `contiguous-layout`. Running the final beacon cell reports progress against the `PyTorch: Contiguous layout` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Contiguous layout` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`contiguous-layout`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "contiguous-layout"
DD_SUBTOPIC = "PyTorch: Contiguous layout"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Contiguous layout — quick refresher

A tensor is **contiguous** iff its stride tuple matches the row-major formula derived from its shape:
```
stride[-1] = 1
stride[k]  = stride[k+1] * shape[k+1]   # walk right-to-left
```
`x.is_contiguous()` checks exactly that. `view` requires it; `reshape` falls back to a copy when it isn't satisfied.

Operations that BREAK contiguity without copying memory:
- `transpose` / `permute` — swap strides only
- `t[::2]` / `expand` — set a stride to 0 or to a multiple of element size
Operations that RESTORE contiguity by copying:
- `.contiguous()` — explicit
- `.reshape(...)` — implicit when needed

### Exercise 3 — classify contiguity from shape and stride tuples

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a `(shape, stride)` pair without instantiating any tensor and return True iff the strides match the row-major contiguous formula.
> Keywords: contiguous, stride-classifier, row-major
> ```

**KCs targeted:** `contiguous-stride-formula`, `is-contiguous-check`

Implement `ex3_is_contiguous_from_meta(shape, stride)`. Given a shape tuple and a stride tuple (both in *elements*, not bytes), return `True` iff a tensor with that metadata would have `is_contiguous() == True`.

**Rules** (handles edge cases the way PyTorch does):
1. Length mismatch ⇒ `False`.
2. The 0-d / 1-d / empty cases: an empty shape `()` is vacuously contiguous (`True`). Any axis of size 0 ⇒ `True` (empty tensors are always contiguous in PyTorch).
3. Otherwise compute the expected row-major strides:
   `expected[-1] = 1`, `expected[k] = expected[k+1] * shape[k+1]`.
4. Compare element-wise. **But:** any axis of size 1 is irrelevant — its stride doesn't matter (PyTorch treats it as a free dimension). Skip the comparison for axes where `shape[k] == 1`.

Inputs: `shape: tuple[int, ...]`, `stride: tuple[int, ...]`.
Output: `bool`.

Do NOT build any tensor. This is a pure-arithmetic predicate.

In [ ]:
def ex3_is_contiguous_from_meta(shape, stride):
    if len(shape) != len(stride):
        return False
    if len(shape) == 0:
        return True
    if any(s == 0 for s in shape):
        return True
    expected_stride = 1
    for k in range(len(shape) - 1, -1, -1):
        if shape[k] != 1:
            if stride[k] != expected_stride:
                return False
        expected_stride *= shape[k]
    return True


<details><summary>Solution</summary>

```python
def ex3_is_contiguous_from_meta(shape, stride):
    if len(shape) != len(stride):
        return False
    if len(shape) == 0:
        return True
    if any(s == 0 for s in shape):
        return True
    expected_stride = 1
    for k in range(len(shape) - 1, -1, -1):
        if shape[k] != 1:
            if stride[k] != expected_stride:
                return False
        expected_stride *= shape[k]
    return True
```

**Right-to-left walk.** The recurrence is `expected[k] = expected[k+1] * shape[k+1]`, so it's natural to start at the last axis (`expected = 1`) and accumulate as you step left.

**Why size-1 axes are free.** Their stride is never multiplied by anything (the only valid index is 0). PyTorch optimizers exploit this to insert size-1 axes without losing contiguity — `x.unsqueeze(0).is_contiguous()` is `True` for any contiguous `x`.

**Why zero-size axes are 'free'.** With an empty axis there's no memory to access, so contiguity is vacuous. This matches PyTorch's behavior (`t.zeros(0, 4).is_contiguous() is True`).

**Reading this predicate is the same skill as predicting `view` errors.** If your tensor came from `transpose` and you can read off its strides, you instantly know whether the next `view` will crash.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()